# 🧠 Brain MRI Tumor Classification — Training & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Danishh-ux/Brain-MRI-model/blob/main/notebooks/Brain_MRI_Training.ipynb)

End-to-end pipeline: data exploration → leakage check → transfer learning (ResNet18 vs EfficientNet-B0) → evaluation with explainability (Grad-CAM), calibration and embedding analysis → export for the Streamlit app.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. Full run takes about 25–35 minutes.

## 1. Setup

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("Brain-MRI-model"):
    !git clone -q https://github.com/Danishh-ux/Brain-MRI-model.git
if IN_COLAB:
    %cd Brain-MRI-model
    !pip install -q kagglehub
!nvidia-smi -L || echo "No GPU detected - training will be slow"

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from brain_mri.config import CLASS_NAMES, DISPLAY_NAMES, TrainConfig, FIGURES_DIR, MODELS_DIR
from brain_mri.data import (list_images, stratified_split, class_counts, find_duplicates, remove_leaked,
                            build_loaders_from_samples, build_transforms, set_seed)
from brain_mri.models import build_model, count_parameters, load_checkpoint, save_checkpoint
from brain_mri.train import train, summarize_history
from brain_mri.evaluate import predict_loader, compute_metrics
from brain_mri import visualize as viz
from brain_mri.gradcam import gradcam_examples

%matplotlib inline
SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| PyTorch", torch.__version__)

## 2. Download the dataset
[Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) — 4 classes, CC BY 4.0. Public datasets download without an API key.

In [ ]:
import kagglehub
DATA_DIR = Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
if not (DATA_DIR / "Training").exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob("Training") if p.is_dir())
print(DATA_DIR, [p.name for p in DATA_DIR.iterdir()])

## 3. Exploratory data analysis

In [ ]:
train_all = list_images(DATA_DIR / "Training")
test = list_images(DATA_DIR / "Testing")
counts = pd.DataFrame({"Training": class_counts(train_all), "Testing": class_counts(test)})
counts.index = [DISPLAY_NAMES[c] for c in counts.index]
counts.loc["Total"] = counts.sum()
display(counts)
viz.plot_class_distribution({"training": class_counts(train_all), "testing": class_counts(test)},
                            FIGURES_DIR / "class_distribution.png");

In [ ]:
viz.plot_sample_grid(train_all, n_per_class=6, path=FIGURES_DIR / "sample_grid.png");

In [ ]:
viz.plot_image_sizes(train_all, path=FIGURES_DIR / "image_sizes.png")
viz.plot_intensity_histograms(train_all, path=FIGURES_DIR / "intensity_histograms.png");

## 4. Data quality: train/test leakage check
This dataset is known to contain near-identical images across splits. Any training image whose perceptual hash matches a test image is removed, so the test score reflects genuinely unseen scans.

In [ ]:
duplicates = find_duplicates(train_all, test)
conflicts = sum(d["label_conflict"] for d in duplicates)
print(f"Test images with a duplicate in training: {len(duplicates)} ({len(duplicates) / len(test):.1%}), "
      f"label conflicts: {conflicts}")
train_clean = remove_leaked(train_all, duplicates)
print(f"Training images: {len(train_all)} -> {len(train_clean)} after removing leaked copies")

In [ ]:
if duplicates:
    from brain_mri.data import load_rgb
    show = duplicates[:4]
    fig, axes = plt.subplots(2, len(show), figsize=(3 * len(show), 6))
    axes = np.array(axes).reshape(2, -1)
    for i, d in enumerate(show):
        axes[0, i].imshow(load_rgb(d["test"])); axes[0, i].set_title("Test", fontsize=10)
        axes[1, i].imshow(load_rgb(d["train"][0])); axes[1, i].set_title("Training duplicate", fontsize=10)
    for ax in axes.ravel(): ax.axis("off")
    fig.suptitle("Examples of leaked duplicates", fontweight="bold"); fig.tight_layout()
    fig.savefig(FIGURES_DIR / "duplicates.png")

## 5. Train / validation / test splits
A stratified 15% **validation** split is carved from the training data. Model selection and early stopping use validation macro F1 only — the test set is touched once, at the very end.

In [ ]:
train_s, val_s = stratified_split(train_clean, 0.15, SEED)
loaders, splits = build_loaders_from_samples(train_s, val_s, test, img_size=224, batch_size=32, num_workers=2)
display(pd.DataFrame({k: class_counts(v) for k, v in splits.items()}).rename(index=DISPLAY_NAMES))
train_tf, eval_tf = build_transforms(224)
viz.plot_augmentations(train_s[0][0], train_tf, path=FIGURES_DIR / "augmentations.png");

## 6. Train and compare models
Both backbones use ImageNet weights, a 1-epoch frozen-backbone warm start, AdamW with a lower backbone learning rate, warmup + cosine schedule, label smoothing, gradient clipping, mixed precision and early stopping.

In [ ]:
ARCHS = ["resnet18", "efficientnet_b0"]
EPOCHS = 20
histories, checkpoints = {}, {}
for arch in ARCHS:
    cfg = TrainConfig(data_dir=str(DATA_DIR), arch=arch, epochs=EPOCHS, batch_size=32, lr=3e-4,
                      patience=6, output_dir=str(MODELS_DIR), seed=SEED)
    _, histories[arch], checkpoints[arch] = train(cfg, loaders=loaders)
    print(summarize_history(histories[arch]), "\n")

In [ ]:
for arch in ARCHS:
    viz.plot_training_curves(histories[arch], f"- {arch}", FIGURES_DIR / f"training_curves_{arch}.png")
plt.show()

## 7. Test-set evaluation

In [ ]:
results, raw = {}, {}
for arch in ARCHS:
    model, meta = load_checkpoint(checkpoints[arch], DEVICE)
    res = predict_loader(model, loaders["test"], DEVICE, tta=True, arch=arch, with_embeddings=True)
    m = compute_metrics(res["probs"], res["labels"])
    m.update(params_m=count_parameters(model) / 1e6, ms_per_image=res["ms_per_image"],
             val_f1=max(histories[arch]["val_f1"]))
    results[arch], raw[arch] = m, res

summary = pd.DataFrame(results).T[["val_f1", "accuracy", "macro_f1", "macro_auc", "cohen_kappa", "ece",
                                   "params_m", "ms_per_image"]].astype(float).round(4)
display(summary)
viz.plot_model_comparison(results, FIGURES_DIR / "model_comparison.png");

In [ ]:
BEST = max(ARCHS, key=lambda a: results[a]["val_f1"])
best_model, best_meta = load_checkpoint(checkpoints[BEST], DEVICE)
best_res, best_metrics = raw[BEST], results[BEST]
print(f"Selected by validation F1: {BEST}")
print(f"Test accuracy {best_metrics['accuracy']:.4f} | macro F1 {best_metrics['macro_f1']:.4f} | "
      f"macro AUC {best_metrics['macro_auc']:.4f} | ECE {best_metrics['ece']:.4f}")
display(pd.DataFrame(best_metrics["report"]).T.round(3))

In [ ]:
viz.plot_confusion_matrices(best_metrics["confusion_matrix"], FIGURES_DIR / "confusion_matrix.png",
                            f"Confusion matrix - {BEST}");

In [ ]:
viz.plot_classification_report(best_metrics["report"], FIGURES_DIR / "classification_report.png")
viz.plot_roc_pr(best_res["probs"], best_res["labels"], FIGURES_DIR / "roc_pr_curves.png");

In [ ]:
viz.plot_calibration(best_res["probs"], best_res["labels"], best_metrics["reliability"], best_metrics["ece"],
                     FIGURES_DIR / "calibration.png");

## 8. Explainability and error analysis

In [ ]:
viz.plot_embeddings(best_res["embeddings"], best_res["labels"], FIGURES_DIR / "tsne_embeddings.png");

In [ ]:
items = gradcam_examples(best_model, best_meta, test, best_res["probs"], best_res["labels"], eval_tf, DEVICE,
                         per_class=3)
viz.plot_gradcam_grid(items, FIGURES_DIR / "gradcam_examples.png");

In [ ]:
viz.plot_misclassified(test, best_res["probs"], best_res["labels"], FIGURES_DIR / "misclassified.png");

## 9. Export for the app
Saves the selected model (with its training history) as `models/best_model.pth`, regenerates `reports/` with the evaluation script, and downloads a zip. Unzip it into the repository root and commit.

In [ ]:
save_checkpoint(MODELS_DIR / "best_model.pth", best_model, BEST, best_meta["img_size"],
                metrics={**best_meta["metrics"], "test_accuracy": best_metrics["accuracy"],
                         "test_macro_f1": best_metrics["macro_f1"]},
                history=histories[BEST])
!"{sys.executable}" scripts/evaluate.py --checkpoint models/best_model.pth --test-dir "{DATA_DIR}/Testing" --tta
(Path("reports") / "model_comparison.json").write_text(json.dumps(
    {a: {k: v for k, v in r.items() if k not in ("reliability", "report", "confusion_matrix")} for a, r in results.items()},
    indent=2))
!zip -qr brain_mri_artifacts.zip models/best_model.pth reports
if IN_COLAB:
    from google.colab import files
    files.download("brain_mri_artifacts.zip")